# Chapter 05 — Classification Models: Live Examples

**Session 2 | Chapter 5 | Duration: ~10 min**

We classify **breast cancer tumors** (malignant vs benign) using multiple classifiers
and visualize their decision boundaries on a 2D slice of the data.

**Key principle:** Different classifiers draw different decision boundaries — the "best" model depends on data geometry and interpretability needs.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

sns.set_theme(style='whitegrid')
np.random.seed(42)
print('Ready!')

## 1. Load the Breast Cancer Dataset

569 samples, 30 features, binary classification: malignant (0) vs benign (1).

In [ ]:
cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = cancer.target  # 0 = malignant, 1 = benign

print('Dataset shape:', X.shape)
print('Class distribution:')
print(pd.Series(y).map({0: 'malignant', 1: 'benign'}).value_counts())
X.head(3)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                      random_state=42, stratify=y)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

## 2. Train and Compare All Classifiers

In [ ]:
classifiers = {
    'Logistic Regression':  Pipeline([('scaler', StandardScaler()),
                                       ('model', LogisticRegression(max_iter=1000))]),
    'KNN (k=5)':            Pipeline([('scaler', StandardScaler()),
                                       ('model', KNeighborsClassifier(n_neighbors=5))]),
    'Decision Tree':        DecisionTreeClassifier(max_depth=4, random_state=42),
    'Random Forest':        RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)':            Pipeline([('scaler', StandardScaler()),
                                       ('model', SVC(kernel='rbf', probability=True))]),
}

results = {}
print(f'{"Model":<22}  {"Accuracy":>10}')
print('-' * 36)
for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    results[name] = acc
    print(f'{name:<22}  {acc:.4f}')

In [ ]:
# Detailed classification report for the best model
best_model_name = max(results, key=results.get)
best_model = classifiers[best_model_name]
y_pred_best = best_model.predict(X_test)

print(f'Best model: {best_model_name} (accuracy={results[best_model_name]:.4f})')
print('\nDetailed classification report:')
print(classification_report(y_test, y_pred_best,
                             target_names=['malignant', 'benign']))

## 3. Visualizing Decision Boundaries

We reduce to 2 features to create a 2D visualization.  
The colored regions show what each model predicts for any new point.

In [ ]:
# Use only 2 features for visualization: worst radius and worst concave points
feat_idx = [cancer.feature_names.tolist().index('worst radius'),
            cancer.feature_names.tolist().index('worst concave points')]
feat_names = [cancer.feature_names[i] for i in feat_idx]

X2 = X.iloc[:, feat_idx].values
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y, test_size=0.2,
                                                          random_state=42, stratify=y)

# Models for 2D visualization
viz_models = {
    'Logistic Regression': Pipeline([('s', StandardScaler()), ('m', LogisticRegression(max_iter=1000))]),
    'KNN (k=5)':           Pipeline([('s', StandardScaler()), ('m', KNeighborsClassifier(5))]),
    'Decision Tree':       DecisionTreeClassifier(max_depth=4, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
}

# Build mesh grid for plotting
h = 0.05
x_min, x_max = X2[:, 0].min() - 1, X2[:, 0].max() + 1
y_min, y_max = X2[:, 1].min() - 0.05, X2[:, 1].max() + 0.05
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
cmap_bg = plt.cm.RdBu
colors = {0: '#e74c3c', 1: '#3498db'}

for ax, (name, clf) in zip(axes.flat, viz_models.items()):
    clf.fit(X2_train, y2_train)
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap=cmap_bg)
    
    for cls, label in [(0, 'malignant'), (1, 'benign')]:
        mask = y2_test == cls
        ax.scatter(X2_test[mask, 0], X2_test[mask, 1],
                   color=colors[cls], label=label, s=40, alpha=0.8,
                   edgecolors='black', linewidths=0.4)
    
    acc = accuracy_score(y2_test, clf.predict(X2_test))
    ax.set_title(f'{name} (2D acc: {acc:.3f})', fontsize=12)
    ax.set_xlabel(feat_names[0])
    ax.set_ylabel(feat_names[1])
    ax.legend(fontsize=9)

plt.suptitle('Decision Boundaries: What Each Model Learned (2D view)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4. Visualizing a Decision Tree

Decision trees are the most interpretable model — every prediction has a traceable path.

In [ ]:
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(18, 7))
plot_tree(dt, feature_names=cancer.feature_names,
          class_names=['malignant', 'benign'],
          filled=True, rounded=True, fontsize=9, ax=ax)
ax.set_title('Decision Tree Classifier (max_depth=3)', fontsize=14)
plt.tight_layout()
plt.show()

print('Each node shows: feature threshold, impurity (gini), sample count, class distribution.')
print('Every path from root to leaf is a human-readable rule!')

## 5. Random Forest Feature Importances

In [ ]:
rf = classifiers['Random Forest']
importances = pd.Series(rf.feature_importances_, index=cancer.feature_names)
top10 = importances.nlargest(10).sort_values()

fig, ax = plt.subplots(figsize=(9, 5))
top10.plot(kind='barh', ax=ax, color='#9b59b6', alpha=0.8)
ax.set_title('Top 10 Most Important Features (Random Forest)', fontsize=13)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()
print('\nThe model relies most heavily on these features to distinguish malignant from benign.')